In [24]:
import os
import sys
import math
import pathlib
import numpy as np
import matplotlib.pyplot as plt


from tqdm import tqdm

import mindspore as ms
from mindspore import  ops
from mindquantum import  Simulator
from mindquantum.core.gates import RY
from mindquantum.core.circuit import Circuit

project_path = pathlib.Path.cwd().parent.parent
sys.path.append(f"{project_path}/src")

from datasets_utils.dataset import get_dataloader_with_idx


In [25]:
n_qubits = 4
n_layers = 0
n_train_samples = 6000
n_test_samples = 3000
batch_size = 200
data_type = 'classification_012'

In [26]:
train_loader_class_dict = {}
for i in range(3):
    train_data_loader_class, test_data_loader_class = get_dataloader_with_idx(n_qubits=n_qubits, n_layers=n_layers,n_train_samples=n_train_samples, n_test_samples=n_test_samples, batch_size=batch_size, data_type=data_type, class_idx=i)
    train_loader_class_dict[i] = train_data_loader_class

In [27]:
train_data_class_dict = {i: [] for i in range(3)}
for i in range(3):
    for data,label in train_loader_class_dict[i]:
        for x in data:
            train_data_class_dict[i].append(x)

In [28]:

def RYEncoder(x):
    sim = Simulator('mqvector', n_qubits)



    circ = Circuit()
    # Ensure theta is a number (float), not a zero-dimensional or single-element array
    for idx, theta in enumerate(x.asnumpy()):
        # Fix: extract scalar from possible array element
        circ += RY(float(theta)).on(idx)
    sim.apply_circuit(circ)
    state = sim.get_qs()
    state = ms.Tensor(state, dtype=ms.complex64)
    state = state.reshape(-1, 1)  # Reshape to column vector
    rho = ops.matmul(state, ops.conj(state.T))
    return rho

In [29]:
train_class_embeddings = {}
for i in range(3):
    count = 0
    rho = np.zeros((2**n_qubits, 2**n_qubits), dtype=np.complex128)
    for data in tqdm(train_data_class_dict[i],desc=f"Processing data for class {i}",leave=False):
        rho += RYEncoder(data)
        count += 1
    train_class_embeddings[i] = rho / count


Processing data for class 0:   0%|          | 0/2000 [00:00<?, ?it/s]/tmp/ipykernel_985624/1304785527.py:10: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  circ += RY(float(theta)).on(idx)


In [17]:
def trace_distance(rho, sigma):
    """Calculate the trace distance between two density matrices"""
    diff = rho - sigma
    eigvals = np.abs(np.linalg.eigvalsh(diff))
    # Calculate trace and divide by 2
    return np.sum(eigvals, axis=-1) / 2

In [31]:
# 将(0, 3, 6)分成两组，一组1个，另一组两个，使得trace distance最大
import numpy as np
from itertools import combinations

# 选定的3个数字
selected_digits = (0,1,2)

max_distance = 0
best_partition = None

# 获取所有可能的1个数字的组合（即单个数字）
for group1 in combinations(selected_digits, 1):
    # 计算第二组（剩余的2个数字）
    group2 = tuple(digit for digit in selected_digits if digit not in group1)
    
    # 计算两组的平均密度矩阵
    avg_density1 = train_class_embeddings[group1[0]]  # 单个数字的密度矩阵
    avg_density2 = sum(train_class_embeddings[digit] for digit in group2) / 2  # 两个数字的平均密度矩阵
    
    # 计算两组之间的trace distance
    distance = trace_distance(avg_density1, avg_density2)
    
    # 更新最大距离
    if distance > max_distance:
        max_distance = distance
        best_partition = (group1, group2)

print(f"最大trace distance: {max_distance}")
print(f"最佳分组方案: {best_partition[0]} 和 {best_partition[1]}")


最大trace distance: 0.7984562792960379
最佳分组方案: (2,) 和 (0, 1)
